# Data Collection: election results (OpenCity)

**Source:** OpenCity (data.opencity.in). The 2014 and 2019 results come from the Trivedi Centre for Political Data (LokDhaba, Ashoka University). The 2024 results come from the Election Commission of India (results.eci.gov.in).

**Collection Date:** 20-10-2026

### 2024 Lok Sabha Election Results
- **Dataset Page:** https://data.opencity.in/dataset/parliamentary-elections-2024-results
- **CSV URL:** https://data.opencity.in/dataset/85a345c6-78c0-4f57-adfc-236c726c5456/resource/d164b73a-b855-4b68-be0c-0f3450e7ab9f/download/1b837c18-4f7a-4acb-aad0-918c51186a54.csv
- **Licence:** Other (Public Domain)

### Historical Lok Sabha Election Results (includes 2014 and 2019)
- **Dataset Page:** https://data.opencity.in/dataset/parliamentary-elections-2019
- **CSV URL:** https://data.opencity.in/dataset/1874a140-a8c4-4d5d-80cd-cbbd400c21e5/resource/ed40f56f-3d97-41a4-8b5d-14b204b6f320/download/81842470-a23e-4785-a22b-ba81284101de.csv
- **Licence:** CC BY-NC-SA 4.0 (non-commercial, share-alike). Credit: Trivedi Centre for Political Data.

**Collection Method:** csv files downloaded programmatically using Python's `requests` library. The notebook skips downloading a file if it already exists locally. Raw files are never edited.

**Purpose:** obtain candidate-level election results, including votes and constituency information, to identify winners and losers.

**Note:** the historical CSV contains election records from many years. It also contains by-elections (`Poll_No` = 1). The year-specific datasets are extracted from this file.

### OpenCity robots.txt

**URL:** https://data.opencity.in/robots.txt

**Findings (checked on the collection date):**
- `User-agent: *` means the rules apply to all crawlers.
- Dataset rating pages, revision pages, dataset history pages, and API endpoints are disallowed.
- `Crawl-Delay: 10` specifies a 10-second delay between crawler requests.

The cell below reads robots.txt again and prints it, so the rules can be quoted in the report.

In [ ]:
# required imports
import requests
import pandas as pd
import os
import time
from pathlib import Path
from urllib.robotparser import RobotFileParser
from bs4 import BeautifulSoup

In [2]:
# dataset urls
URL_2024 = "https://data.opencity.in/dataset/85a345c6-78c0-4f57-adfc-236c726c5456/resource/d164b73a-b855-4b68-be0c-0f3450e7ab9f/download/1b837c18-4f7a-4acb-aad0-918c51186a54.csv"
URL_HIST = "https://data.opencity.in/dataset/1874a140-a8c4-4d5d-80cd-cbbd400c21e5/resource/ed40f56f-3d97-41a4-8b5d-14b204b6f320/download/81842470-a23e-4785-a22b-ba81284101de.csv"

COLLECTION_DATE = "2026-10-20"
USER_AGENT = "IITM-DA-coursework-bot/1.0 (student project; contact: da25b051@smail.iitm.ac.in)"

In [3]:
# read and print robots.txt, check our urls against it
robots_url = "https://data.opencity.in/robots.txt"

try:
    robots_text = requests.get(robots_url, headers={"User-Agent": USER_AGENT}, timeout=30).text
    print(robots_text)

    rp = RobotFileParser()
    rp.parse(robots_text.splitlines())

    for url in [URL_2024, URL_HIST]:
        print("Allowed:", rp.can_fetch(USER_AGENT, url))
    CRAWL_DELAY = rp.crawl_delay(USER_AGENT) or 10
except Exception as error:
    # fall back to the delay found manually on the collection date
    print("Could not read robots.txt now:", error)
    CRAWL_DELAY = 10

print("\nCrawl delay used (seconds):", CRAWL_DELAY)

User-agent: *
Disallow: /dataset/rate/
Disallow: /revision/
Disallow: /dataset/*/history
Disallow: /api/
Crawl-Delay: 10



Allowed: True
Allowed: True

Crawl delay used (seconds): 10


In [4]:
# licence and terms: print the licence line from each dataset page so it can be quoted in the report
for page in ["https://data.opencity.in/dataset/parliamentary-elections-2024-results",
             "https://data.opencity.in/dataset/parliamentary-elections-2019"]:
    try:
        text = requests.get(page, headers={"User-Agent": USER_AGENT}, timeout=30).text
        soup = BeautifulSoup(text, "html.parser")
        print(page)
        print("  licence:", soup.find(string=lambda t: t and "License" in t))
    except Exception as error:
        print("Could not read", page, error)

# initialize session
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

# folders (raw files are never edited, only added)
raw_dir = Path("../raw")
interim_dir = Path("../interim")
raw_dir.mkdir(exist_ok=True)
interim_dir.mkdir(exist_ok=True)

https://data.opencity.in/dataset/parliamentary-elections-2024-results
  licence:  License
https://data.opencity.in/dataset/parliamentary-elections-2019
  licence:  License


In [ ]:
# download the datasets if you haven't (file names carry source and date)
datasets = {
    "2024": (URL_2024, raw_dir / f"opencity_results2024_{COLLECTION_DATE}.csv"),
    "historical": (URL_HIST, raw_dir / f"opencity_results_historical_{COLLECTION_DATE}.csv")
}

for name, (url, csv_path) in datasets.items():
    if csv_path.exists():
        print(f"{name} dataset already exists, skipping download")
        continue

    response = session.get(url, timeout=120)
    response.raise_for_status()
    csv_path.write_bytes(response.content)
    print(f"{name} dataset downloaded successfully!!")

    # delay
    time.sleep(CRAWL_DELAY)

2024 dataset downloaded successfully!!
historical dataset downloaded successfully!!


,file,bytes,sha256
0,opencity_results2024_2026-10-20.csv,724479,1a0c52ff42e06d08ce80b2d896b69409fe00cbaf2d015e...
1,opencity_tcpd_historical_2026-10-20.csv,20469716,a3dbeb280776ac6a59cb0228588119c0721d3e429c924e...


### Raw file 1: 2024 results

Columns kept in the analysis and what they mean:

| Column | Meaning |
|---|---|
| State | State or union territory name |
| PC No | Parliamentary constituency number (ECI) |
| PC Name | Parliamentary constituency name |
| Candidate | Candidate name in capital letters, as printed by ECI |
| Party | Full party name as registered with ECI. `Independent` and `None of the Above` (NOTA) are included |
| Total Votes | EVM votes + postal votes (count of votes) |
| Vote Share | Percentage of all valid votes in the constituency, stored as text |

`Sl no`, `EVM Votes` and `Postal Votes` are not needed for the analysis.

In [6]:
# load 2024 data
df_2024 = pd.read_csv(datasets["2024"][1])

# light data inspection
print("First 5 rows:\n", df_2024.head())
print("\nShape:", df_2024.shape)
print("\nColumns:", df_2024.columns)
print("\nData types:\n", df_2024.dtypes)

First 5 rows:
             State  PC No     PC Name  Sl no                Candidate  \
0  Andhra Pradesh      1  Araku (ST)      1       GUMMA THANUJA RANI   
1  Andhra Pradesh      1  Araku (ST)      2        KOTHAPALLI GEETHA   
2  Andhra Pradesh      1  Araku (ST)      3  APPALANARASA PACHIPENTA   
3  Andhra Pradesh      1  Araku (ST)      4   AVASHYA LAHARI . VARAM   
4  Andhra Pradesh      1  Araku (ST)      5    SAMAREDDY BALAKRISHNA   

                                   Party EVM Votes Postal Votes Total Votes  \
0  Yuvajana Sramika Rythu Congress Party    471470         5535      477005   
1                 Bharatiya Janata Party    417113         9312      426425   
2    Communist Party of India  (Marxist)    119016         4113      123129   
3                    Bahujan Samaj Party     24858          892       25750   
4                            Independent      9493           42        9535   

  Vote Share  
0      40.96  
1      36.62  
2      10.57  
3       2.21  
4 

### Raw file 2: historical results (2014 and 2019 are inside)

Columns kept in the analysis and what they mean:

| Column | Meaning |
|---|---|
| State_Name | State name with underscores, e.g. `Tamil_Nadu` |
| Constituency_No, Constituency_Name | Parliamentary constituency number and name |
| Year, Poll_No | Election year. `Poll_No` is 0 for the general election and 1 or more for by-elections |
| Candidate, Sex | Candidate name in capital letters, and M/F |
| Party | Party abbreviation (e.g. BJP, INC, IND, NOTA) |
| Position | Finishing position as published |
| Votes, Vote_Share_Percentage | Votes (count) and share of valid votes (percent) |
| Margin | Votes between this candidate and the next one (count) |
| Constituency_Type | GEN, SC or ST |
| Incumbent, Turncoat | True if the candidate was the sitting MP, or changed party since the last contest |
| Electors | Registered voters in the constituency (count) |

In [7]:
# load historical data
df_hist = pd.read_csv(datasets["historical"][1], low_memory=False)

# light data inspection
print("First 5 rows:\n", df_hist.head())
print("\nShape:", df_hist.shape)
print("\nData types:\n", df_hist.dtypes.to_string())

First 5 rows:
                   State_Name  Assembly_No  Constituency_No  Year  month  \
0  Andaman_&_Nicobar_Islands           17                1  2019    4.0   
1  Andaman_&_Nicobar_Islands           17                1  2019    4.0   
2  Andaman_&_Nicobar_Islands           17                1  2019    4.0   
3  Andaman_&_Nicobar_Islands           17                1  2019    4.0   
4  Andaman_&_Nicobar_Islands           17                1  2019    4.0   

   Poll_No  DelimID  Position              Candidate Sex  ... No_Terms  \
0        0        4         1     KULDEEP RAI SHARMA   M  ...      1.0   
1        0        4         2           VISHAL JOLLY   M  ...      0.0   
2        0        4         3  PARITOSH KUMAR HALDAR   M  ...      0.0   
3        0        4         4         SANJAY MESHACK   M  ...      0.0   
4        0        4         5           PRAKASH MINJ   M  ...      0.0   

   Turncoat Incumbent  Recontest       MyNeta_education  \
0     False     False       Tr

In [8]:
# further inspect the historical dataset since there are many more rows compared to the 2024 dataset
print("Election years (last 6):\n", df_hist["Year"].value_counts().sort_index().tail(6))
print("\nElection types:\n", df_hist["Election_Type"].value_counts())
print("\nPoll_No for 2014 and 2019:\n", df_hist[df_hist["Year"].isin([2014, 2019])].groupby(["Year", "Poll_No"]).size())

Election years (last 6):
 Year
2016      41
2017      14
2018     150
2019    8613
2020       8
2021      89
Name: count, dtype: int64

Election types:
 Election_Type
Lok Sabha Election (GE)    91669
Name: count, dtype: int64

Poll_No for 2014 and 2019:
 Year  Poll_No
2014  0          8794
      1            49
2019  0          8596
      1            17
dtype: int64


In [9]:
# extract seperate datasets for the 2014 and 2019 elections
# by-elections (Poll_No > 0) are kept here, they are removed later in data prep with a count
df_2014 = df_hist[df_hist["Year"] == 2014].copy()
df_2019 = df_hist[df_hist["Year"] == 2019].copy()

for year, df in [(2014, df_2014), (2019, df_2019)]:
    print(f"{year} shape:", df.shape)

2014 shape: (8843, 45)
2019 shape: (8613, 45)


In [10]:
# by-elections held after 2014 (Poll_No > 0), needed later to find sitting MPs who entered between general elections
df_bye = df_hist[(df_hist["Poll_No"] > 0) & (df_hist["Year"] >= 2014)].copy()
print("by-election rows from 2014 on:", df_bye.shape[0], "| years:", sorted(df_bye["Year"].unique()))
df_bye.to_csv(interim_dir / "opencity_byelections_2014_onwards.csv", index=False)

# save year-wise raw extracts to interim
for year, df in [(2014, df_2014), (2019, df_2019), (2024, df_2024)]:
    df.to_csv(interim_dir / f"opencity_electiondata_{year}.csv", index=False)

print("year-wise data sets saved successfully!")

by-election rows from 2014 on: 410 | years: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
year-wise data sets saved successfully!
